In [1]:
import random
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence
from sklearn.model_selection import train_test_split

random.seed(42)
torch.manual_seed(42)

PAD_TOKEN = "<PAD>"
BOS_TOKEN = "<BOS>"
EOS_TOKEN = "<EOS>"
UNK_TOKEN = "<UNK>"

PAD_IDX = 0
BOS_IDX = 1
EOS_IDX = 2
UNK_IDX = 3

In [2]:
class TranslationDataset(Dataset):
    def __init__(self, pairs, input_vocab=None, output_vocab=None):
        self.pairs = pairs

        if input_vocab is None or output_vocab is None:
            self.input_vocab, self.output_vocab = self.build_vocabs(pairs)
        else:
            self.input_vocab = input_vocab
            self.output_vocab = output_vocab

        self.input_vocab_inv = {v: k for k, v in self.input_vocab.items()}
        self.output_vocab_inv = {v: k for k, v in self.output_vocab.items()}

    @staticmethod
    def build_vocabs(pairs):
        input_vocab = {PAD_TOKEN: PAD_IDX, BOS_TOKEN: BOS_IDX, EOS_TOKEN: EOS_IDX, UNK_TOKEN: UNK_IDX}
        output_vocab = {PAD_TOKEN: PAD_IDX, BOS_TOKEN: BOS_IDX, EOS_TOKEN: EOS_IDX, UNK_TOKEN: UNK_IDX}

        for eng, rus in pairs:
            for word in eng.split():
                if word not in input_vocab:
                    input_vocab[word] = len(input_vocab)
            for word in rus.split():
                if word not in output_vocab:
                    output_vocab[word] = len(output_vocab)

        return input_vocab, output_vocab

    def __len__(self):
        return len(self.pairs)

    def encode_sentence(self, sentence, vocab, add_bos_eos=False):
        words = sentence.lower().split()
        tokens = [vocab.get(w, vocab[UNK_TOKEN]) for w in words]
        if add_bos_eos:
            tokens = [vocab[BOS_TOKEN]] + tokens + [vocab[EOS_TOKEN]]
        return torch.tensor(tokens, dtype=torch.long)

    def __getitem__(self, idx):
        eng, rus = self.pairs[idx]
        input_seq = self.encode_sentence(eng, self.input_vocab, add_bos_eos=True)
        target_seq = self.encode_sentence(rus, self.output_vocab, add_bos_eos=True)
        return input_seq, target_seq


def collate_fn(batch):
    input_seqs, target_seqs = zip(*batch)
    input_padded = pad_sequence(input_seqs, batch_first=True, padding_value=PAD_IDX)
    target_padded = pad_sequence(target_seqs, batch_first=True, padding_value=PAD_IDX)
    return input_padded, target_padded

In [3]:
def load_data(filename="rus.txt", n_samples=10000):
    df = pd.read_csv(filename, sep="\t", header=None, on_bad_lines="skip")

    if len(df.columns) >= 2:
        df = df.iloc[:, :2]
        df.columns = ["en", "ru"]

    df = df.head(n_samples).copy()
    df["en"] = df["en"].astype(str).str.lower()
    df["ru"] = df["ru"].astype(str).str.lower()

    pairs = list(zip(df["en"], df["ru"]))
    return pairs


In [4]:
def build_datasets(pairs, test_size=0.2, val_size=0.1):
    train_pairs, temp_pairs = train_test_split(pairs, test_size=test_size, random_state=42)
    relative_val_size = val_size / (1.0 - test_size)
    val_pairs, test_pairs = train_test_split(temp_pairs, test_size=relative_val_size, random_state=42)
    return train_pairs, val_pairs, test_pairs

In [5]:
class Seq2Seq(nn.Module):
    def __init__(self, input_size, output_size, embedding_dim=256, hidden_size=512, num_layers=1, cell_type="gru"):
        super().__init__()
        self.hidden_size = hidden_size
        self.num_layers = num_layers
        self.cell_type = cell_type

        self.encoder_emb = nn.Embedding(input_size, embedding_dim, padding_idx=PAD_IDX)
        self.decoder_emb = nn.Embedding(output_size, embedding_dim, padding_idx=PAD_IDX)

        if cell_type == "gru":
            self.encoder_rnn = nn.GRU(embedding_dim, hidden_size, num_layers, batch_first=True)
            self.decoder_rnn = nn.GRU(embedding_dim, hidden_size, num_layers, batch_first=True)
        elif cell_type == "lstm":
            self.encoder_rnn = nn.LSTM(embedding_dim, hidden_size, num_layers, batch_first=True)
            self.decoder_rnn = nn.LSTM(embedding_dim, hidden_size, num_layers, batch_first=True)
        else:
            raise ValueError("cell_type must be 'gru' or 'lstm'.")

        self.fc = nn.Linear(hidden_size, output_size)

    def encode(self, input_seq, input_lengths=None):
        enc_emb = self.encoder_emb(input_seq)
        if input_lengths is not None:
            packed = nn.utils.rnn.pack_padded_sequence(enc_emb, input_lengths.cpu(), batch_first=True, enforce_sorted=False)
            if self.cell_type == "gru":
                _, enc_hidden = self.encoder_rnn(packed)
            else:
                _, (enc_hidden, enc_cell) = self.encoder_rnn(packed)
                return enc_hidden, enc_cell
        else:
            if self.cell_type == "gru":
                _, enc_hidden = self.encoder_rnn(enc_emb)
            else:
                _, (enc_hidden, enc_cell) = self.encoder_rnn(enc_emb)
                return enc_hidden, enc_cell

        return enc_hidden

    def decode_step(self, dec_input, hidden, cell=None):
        dec_emb = self.decoder_emb(dec_input)
        if self.cell_type == "gru":
            dec_out, hidden = self.decoder_rnn(dec_emb, hidden)
            return dec_out, hidden, None
        else:
            dec_out, (hidden, cell) = self.decoder_rnn(dec_emb, (hidden, cell))
            return dec_out, hidden, cell

    def forward(self, input_seq, target_seq=None, teacher_forcing_ratio=1.0, input_lengths=None):
        if self.cell_type == "lstm":
            enc_hidden, enc_cell = self.encode(input_seq, input_lengths=input_lengths)
            hidden = enc_hidden
            cell = enc_cell
        else:
            hidden = self.encode(input_seq, input_lengths=input_lengths)
            cell = None

        if target_seq is not None:
            target_len = target_seq.size(1) - 1
            outputs = []
            dec_input = target_seq[:, 0:1]

            for t in range(target_len):
                dec_out, hidden, cell = self.decode_step(dec_input, hidden, cell)
                out = self.fc(dec_out)
                outputs.append(out)

                if random.random() < teacher_forcing_ratio:
                    dec_input = target_seq[:, t + 1:t + 2]
                else:
                    dec_input = out.argmax(dim=2)

            return torch.cat(outputs, dim=1)
        else:
            batch_size = input_seq.size(0)
            device = input_seq.device
            dec_input = torch.full((batch_size, 1), BOS_IDX, dtype=torch.long, device=device)
            outputs = []

            for _ in range(30):
                dec_out, hidden, cell = self.decode_step(dec_input, hidden, cell)
                out = self.fc(dec_out)
                outputs.append(out)
                dec_input = out.argmax(dim=2)

            return torch.cat(outputs, dim=1)

In [6]:
def teacher_forcing_schedule(epoch, epochs, start=1.0, end=0.2, warmup_frac=0.2, decay_frac=0.6):
    warmup_epochs = int(epochs * warmup_frac)
    decay_epochs = int(epochs * decay_frac)

    if epoch < warmup_epochs:
        return start
    elif epoch < warmup_epochs + decay_epochs:
        progress = (epoch - warmup_epochs) / max(1, decay_epochs)
        return start - (start - end) * progress
    else:
        return end

In [7]:
def make_loader(dataset, batch_size=32, shuffle=True):
    return DataLoader(dataset, batch_size=batch_size, shuffle=shuffle, collate_fn=collate_fn)

In [8]:
def compute_loss(output, target_seq, criterion):
    return criterion(output.reshape(-1, output.size(-1)), target_seq[:, 1:].reshape(-1))

In [9]:
def train_one_epoch(model, loader, optimizer, criterion, device, teacher_forcing_ratio):
    model.train()
    total_loss = 0.0

    for input_seq, target_seq in loader:
        input_seq = input_seq.to(device)
        target_seq = target_seq.to(device)
        input_lengths = (input_seq != PAD_IDX).sum(dim=1)

        optimizer.zero_grad()
        output = model(input_seq, target_seq, teacher_forcing_ratio=teacher_forcing_ratio, input_lengths=input_lengths)
        loss = compute_loss(output, target_seq, criterion)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        total_loss += loss.item()

    return total_loss / max(1, len(loader))

In [10]:
@torch.no_grad()
def evaluate_loss(model, loader, criterion, device):
    model.eval()
    total_loss = 0.0

    for input_seq, target_seq in loader:
        input_seq = input_seq.to(device)
        target_seq = target_seq.to(device)
        input_lengths = (input_seq != PAD_IDX).sum(dim=1)

        output = model(input_seq, target_seq, teacher_forcing_ratio=0.0, input_lengths=input_lengths)
        loss = compute_loss(output, target_seq, criterion)
        total_loss += loss.item()

    return total_loss / max(1, len(loader))

In [11]:
def train_model(train_pairs, val_pairs, config, epochs=60, batch_size=64):
    device = torch.device("mps" if torch.backends.mps.is_available() else "cuda" if torch.cuda.is_available() else "cpu")

    train_dataset = TranslationDataset(train_pairs)
    val_dataset = TranslationDataset(val_pairs, input_vocab=train_dataset.input_vocab, output_vocab=train_dataset.output_vocab)

    train_loader = make_loader(train_dataset, batch_size=batch_size, shuffle=True)
    val_loader = make_loader(val_dataset, batch_size=batch_size, shuffle=False)

    model = Seq2Seq(
        input_size=len(train_dataset.input_vocab),
        output_size=len(train_dataset.output_vocab),
        embedding_dim=config["embedding_dim"],
        hidden_size=config["hidden_size"],
        num_layers=config["layers"],
        cell_type=config["cell_type"]
    ).to(device)

    optimizer = optim.Adam(model.parameters(), lr=config["lr"])
    criterion = nn.CrossEntropyLoss(ignore_index=PAD_IDX)

    history = []

    print(f"Обучение модели {config['name']}")

    for epoch in range(epochs):
        tf_ratio = teacher_forcing_schedule(epoch, epochs, start=1.0, end=config["tf_end"], warmup_frac=0.2, decay_frac=0.6)
        train_loss = train_one_epoch(model, train_loader, optimizer, criterion, device, tf_ratio)
        val_loss = evaluate_loss(model, val_loader, criterion, device)

        history.append({
            "epoch": epoch + 1,
            "train_loss": train_loss,
            "val_loss": val_loss,
            "teacher_forcing_ratio": tf_ratio
        })

        if (epoch + 1) % 5 == 0 or epoch == 0:
            print(f"Epoch {epoch+1}/{epochs}, Train Loss: {train_loss:.4f}, Val Loss: {val_loss:.4f}, TF: {tf_ratio:.3f}")

    return model, train_dataset, pd.DataFrame(history), device

In [12]:
@torch.no_grad()
def translate(model, dataset, sentence, max_len=30):
    device = next(model.parameters()).device
    model.eval()

    words = sentence.lower().split()
    input_seq = [dataset.input_vocab.get(w, dataset.input_vocab[UNK_TOKEN]) for w in words]
    input_seq = [dataset.input_vocab[BOS_TOKEN]] + input_seq + [dataset.input_vocab[EOS_TOKEN]]
    input_seq = torch.tensor([input_seq], dtype=torch.long, device=device)
    input_lengths = torch.tensor([input_seq.size(1)], dtype=torch.long, device=device)

    if model.cell_type == "lstm":
        hidden, cell = model.encode(input_seq, input_lengths=input_lengths)
    else:
        hidden = model.encode(input_seq, input_lengths=input_lengths)
        cell = None

    dec_input = torch.tensor([[dataset.output_vocab[BOS_TOKEN]]], dtype=torch.long, device=device)
    output_words = []

    for _ in range(max_len):
        dec_out, hidden, cell = model.decode_step(dec_input, hidden, cell)
        out = model.fc(dec_out)
        next_token = out.argmax(dim=2).item()

        if next_token == dataset.output_vocab[EOS_TOKEN]:
            break
        if next_token != dataset.output_vocab[PAD_TOKEN]:
            output_words.append(dataset.output_vocab_inv.get(next_token, UNK_TOKEN))

        dec_input = torch.tensor([[next_token]], dtype=torch.long, device=device)

    return " ".join(output_words)

In [13]:
def evaluate_examples(model, dataset, sentences):
    results = []
    for s in sentences:
        results.append((s, translate(model, dataset, s)))
    return results

In [14]:
pairs = load_data("rus.txt", n_samples=10000)
train_pairs, val_pairs, test_pairs = build_datasets(pairs, test_size=0.2, val_size=0.1)

configs = [
    {"name": "GRU 1 слой", "layers": 1, "cell_type": "gru", "embedding_dim": 256, "hidden_size": 512, "lr": 0.001, "tf_end": 0.3},
    {"name": "GRU 2 слоя", "layers": 2, "cell_type": "gru", "embedding_dim": 256, "hidden_size": 512, "lr": 0.001, "tf_end": 0.3},
    {"name": "LSTM 1 слой", "layers": 1, "cell_type": "lstm", "embedding_dim": 256, "hidden_size": 512, "lr": 0.001, "tf_end": 0.3},
    {"name": "LSTM 2 слоя", "layers": 2, "cell_type": "lstm", "embedding_dim": 256, "hidden_size": 512, "lr": 0.001, "tf_end": 0.3},
]

all_results = {}
all_histories = {}

for config in configs:
    model, dataset, history, device = train_model(train_pairs, val_pairs, config, epochs=60, batch_size=64)

    print(f"\n{config['name']}:")
    test_sentences = ["hello", "how are you", "good night", "how is it gong", "what is wrong"]
    for sent in test_sentences:
        translation = translate(model, dataset, sent)
        print(f"{sent} -> {translation}")

    all_results[config["name"]] = {
        "model": model,
        "dataset": dataset,
        "device": device
    }
    all_histories[config["name"]] = history

summary_rows = []
for name, data in all_results.items():
    model = data["model"]
    dataset = data["dataset"]
    device = data["device"]
    test_dataset = TranslationDataset(test_pairs, input_vocab=dataset.input_vocab, output_vocab=dataset.output_vocab)
    test_loader = make_loader(test_dataset, batch_size=64, shuffle=False)
    criterion = nn.CrossEntropyLoss(ignore_index=PAD_IDX)
    test_loss = evaluate_loss(model, test_loader, criterion, device)

    summary_rows.append({
        "model": name,
        "test_loss": test_loss
    })

summary_df = pd.DataFrame(summary_rows)
print("\nФинальная оценка модели:")
print(summary_df.sort_values("test_loss"))

Обучение модели GRU 1 слой
Epoch 1/60, Train Loss: 4.6646, Val Loss: 4.4058, TF: 1.000
Epoch 5/60, Train Loss: 0.8816, Val Loss: 3.6367, TF: 1.000
Epoch 10/60, Train Loss: 0.5480, Val Loss: 3.8713, TF: 1.000
Epoch 15/60, Train Loss: 0.5248, Val Loss: 3.9199, TF: 0.961
Epoch 20/60, Train Loss: 0.5458, Val Loss: 3.6926, TF: 0.864
Epoch 25/60, Train Loss: 0.5789, Val Loss: 3.6839, TF: 0.767
Epoch 30/60, Train Loss: 0.5776, Val Loss: 3.6355, TF: 0.669
Epoch 35/60, Train Loss: 0.5776, Val Loss: 3.5764, TF: 0.572
Epoch 40/60, Train Loss: 0.5614, Val Loss: 3.6244, TF: 0.475
Epoch 45/60, Train Loss: 0.5669, Val Loss: 3.6126, TF: 0.378
Epoch 50/60, Train Loss: 0.5578, Val Loss: 3.6919, TF: 0.300
Epoch 55/60, Train Loss: 0.5453, Val Loss: 3.7422, TF: 0.300
Epoch 60/60, Train Loss: 0.5248, Val Loss: 3.8199, TF: 0.300

GRU 1 слой:
hello -> привет
how are you -> как ты
good night -> мы будем
how is it gong -> вот это
what is wrong -> вот и
Обучение модели GRU 2 слоя
Epoch 1/60, Train Loss: 4.7202, 